[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Farid841/pre_processing-container-generator-from-mlflow/blob/main/docs/tutorial.ipynb)


In [ ]:
# Installer les dépendances (Colab / environnement vierge)
!pip install -q mlflow scikit-learn requests numpy pandas

# MLflow x Fink Quickstart

Train a classifier on ZTF alerts and deploy it on **Fink** in 6 steps.

Prerequisites:
```bash
pip install mlflow scikit-learn requests numpy pandas
export MLFLOW_TRACKING_URI=http://127.0.0.1:5000
```


## Step 1: Configure MLflow


In [ ]:
import getpass
import os


import mlflow

tracking_uri = (
    os.environ.get("MLFLOW_TRACKING_URI")
    or input("MLFLOW_TRACKING_URI [https://mlflow-dev.fink-broker.org]: ")
    or "https://mlflow-dev.fink-broker.org"
)
username = os.environ.get("MLFLOW_TRACKING_USERNAME") or input(
    "MLFLOW_TRACKING_USERNAME (leave empty if not required): "
)
password = os.environ.get("MLFLOW_TRACKING_PASSWORD") or getpass.getpass(
    "MLFLOW_TRACKING_PASSWORD (leave empty if not required): "
)

if username:
    os.environ["MLFLOW_TRACKING_USERNAME"] = username
if password:
    os.environ["MLFLOW_TRACKING_PASSWORD"] = password

mlflow.set_tracking_uri(tracking_uri)
print(f"MLflow URI  : {tracking_uri}")
print(f"User        : {username or '(none)'}")


## Step 2: Load real ZTF alerts


In [ ]:
import numpy as np
import pandas as pd
import requests
from sklearn.model_selection import train_test_split

FINK_API = "https://api.ztf.fink-portal.org/api/v1/latests"
COLS = (
    "i:objectId,i:rb,i:drb,i:classtar,i:magpsf,i:sigmapsf,"
    "i:diffmaglim,i:ndethist,i:isdiffpos,i:sgscore1,i:distpsnr1"
)


def fetch(class_name, n):
    r = requests.post(
        FINK_API,
        json={"class": class_name, "n": n, "columns": COLS, "format": "json"},
        timeout=15,
    )
    r.raise_for_status()
    return r.json()


def to_alert(row):
    c = {k[2:]: v for k, v in row.items() if k.startswith("i:")}
    return {"candidate": c}


try:
    real_rows = fetch("SN candidate", 200)
    bogus_rows = fetch("Unknown", 200)
    alerts = [to_alert(r) for r in real_rows] + [to_alert(r) for r in bogus_rows]
    labels = [1] * len(real_rows) + [0] * len(bogus_rows)
    data_source = f"Fink Portal | {len(real_rows)} SN candidates + {len(bogus_rows)} Unknown"
except Exception as e:
    print(f"API unreachable ({e}), using simulated data...")
    

In [ ]:
sample = pd.DataFrame([a["candidate"] for a in alerts[:5]]).round(3)
sample.insert(0, "label", labels[:5])
display(sample)

## Step 3: Write the preprocessing function


In [ ]:
def pre_processing(alert: dict) -> list:
    """Extract 18 features from a ZTF alert (schema 3.3)."""
    c = alert.get("candidate") or {}
    prv = alert.get("prv_candidates") or []

    _dp = c.get("isdiffpos")
    isdiffpos = 1.0 if _dp in ("t", "1", 1, True) else 0.0

    mags, jds = [], []
    for p in prv:
        if not isinstance(p, dict):
            continue
        if p.get("isdiffpos") not in ("t", "1", 1, True):
            continue
        m, j = p.get("magpsf"), p.get("jd")
        if m is not None:
            try:
                mags.append(float(m))
            except (TypeError, ValueError):
                pass
        if j is not None:
            try:
                jds.append(float(j))
            except (TypeError, ValueError):
                pass
    n_prev_det = float(len(mags))
    if len(mags) >= 2:
        _mean = sum(mags) / len(mags)
        mag_std = (sum((_m - _mean) ** 2 for _m in mags) / len(mags)) ** 0.5
    else:
        mag_std = 0.0
    time_baseline = (max(jds) - min(jds)) if len(jds) >= 2 else 0.0

    def _sf(v):
        if v is None:
            return 0.0
        try:
            return float(v)
        except (TypeError, ValueError):
            return 0.0

    return [
        _sf(c.get("rb")),
        _sf(c.get("drb")),
        _sf(c.get("classtar")),
        _sf(c.get("fwhm")),
        _sf(c.get("elong")),
        _sf(c.get("magpsf")),
        _sf(c.get("sigmapsf")),
        _sf(c.get("diffmaglim")),
        _sf(c.get("ndethist")),
        _sf(c.get("scorr")),
        _sf(c.get("chinr")),
        _sf(c.get("sharpnr")),
        _sf(c.get("sgscore1")),
        _sf(c.get("distpsnr1")),
        isdiffpos,
        n_prev_det,
        mag_std,
        time_baseline,
    ]


FEATURE_NAMES = [
    "rb", "drb", "classtar", "fwhm", "elong",
    "magpsf", "sigmapsf", "diffmaglim", "ndethist", "scorr",
    "chinr", "sharpnr", "sgscore1", "distpsnr1", "isdiffpos",
    "n_prev_det", "mag_std", "time_baseline",
]

X = np.array([pre_processing(a) for a in alerts])
y = np.array(labels)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X shape : {X.shape}")
pd.DataFrame(X[:5], columns=FEATURE_NAMES).round(3)


## Step 4: Train the classifier


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    class_weight="balanced",
    random_state=42,
)
model.fit(X_train, y_train)

acc = accuracy_score(y_test, model.predict(X_test))
f1  = f1_score(y_test, model.predict(X_test))
print(f"Accuracy : {acc:.1%}")
print(f"F1-score : {f1:.1%}")

## Step 5: Log the model and `preprocessing.py`

A `with mlflow.start_run()` block delimits the run. We log parameters,
metrics, the model, and `preprocessing.py` (the file Fink executes
on each alert) in the same block.


In [ ]:
from sklearn.metrics import precision_score, recall_score

mlflow.set_experiment("fink-bang_bang")

with mlflow.start_run(run_name="ztf-real-bogus") as run:
    mlflow.log_params({
        "n_estimators": model.n_estimators,
        "max_depth": model.max_depth,
        "features": str(FEATURE_NAMES),
    })

    y_pred = model.predict(X_test)
    mlflow.log_metrics({
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "f1_score": float(f1_score(y_test, y_pred)),
        "precision": float(precision_score(y_test, y_pred)),
        "recall": float(recall_score(y_test, y_pred)),
    })

    model_info = mlflow.sklearn.log_model(
        model,
        name="model",
        registered_model_name="ztf-bang_bang",
    )

    mlflow.log_artifact("preprocessing.py", artifact_path="preprocessing")
    mlflow.set_tag("type", "real-bogus")

print(f"Run ID   : {run.info.run_id}")
print(f"Model URI: {model_info.model_uri}")


## Step 6: Load the model and predict


In [ ]:
test_alerts = alerts[:5]
X_demo = np.array([pre_processing(a) for a in test_alerts])

loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)
scores = loaded_model.predict(X_demo)

result = pd.DataFrame(X_demo, columns=FEATURE_NAMES).round(3)
result["score"] = scores.round(4)
result["label"] = result["score"].apply(lambda s: "real ✓" if s >= 0.5 else "bogus ✗")
display(result)

## Step 7: Read results produced by Fink

Once your model is deployed, predictions arrive in a `fink_ai_*` topic:

```bash
fink_datainference \
    -topic fink_ai_2026-06-08_ztf-real-bogus \
    -servers kafka.fink-broker.org:9093 \
    -outdir ./my_results
```


In [ ]:
import random as _rnd

rng7 = _rnd.Random(55)


def beta7(a, b):
    x, y = rng7.gammavariate(a, 1), rng7.gammavariate(b, 1)
    return max(0.0, min(1.0, x / (x + y)))


rows = []
for i in range(15):
    is_real = rng7.random() > 0.45
    x_vec = np.array([[
        beta7(8, 2) if is_real else beta7(2, 8),                    # rb
        beta7(8, 2) if is_real else beta7(2, 8),                    # drb
        rng7.uniform(0, 1),                                          # classtar
        abs(rng7.gauss(2.2, 0.3) if is_real else rng7.gauss(3.5, 0.8)),  # fwhm
        max(1.0, rng7.gauss(1.05, 0.05) if is_real else rng7.gauss(1.3, 0.3)),  # elong
        rng7.gauss(19.0, 2.0),                                       # magpsf
        abs(rng7.gauss(0.15, 0.05) if is_real else rng7.gauss(0.3, 0.1)),  # sigmapsf
        rng7.gauss(20.0, 1.0),                                       # diffmaglim
        float(max(1, int(rng7.gauss(8, 3))) if is_real else max(0, int(rng7.gauss(1, 1)))),  # ndethist
        rng7.gauss(15.0, 3.0) if is_real else rng7.gauss(5.0, 2.0), # scorr
        abs(rng7.gauss(1.0, 0.15)),                                  # chinr
        rng7.gauss(0.0, 0.1),                                        # sharpnr
        rng7.uniform(0.0, 0.3) if is_real else rng7.uniform(0.7, 1.0),  # sgscore1
        rng7.uniform(0.1, 3.0),                                      # distpsnr1
        1.0,                                                          # isdiffpos
        0.0, 0.0, 0.0,                                               # n_prev_det, mag_std, time_baseline
    ]])
    score = float(model.predict_proba(x_vec)[0][1])
    rows.append({
        "objectId": f"ZTF2{'1' if is_real else '2'}{i:07d}",
        "candid": rng7.randint(10**9, 10**10 - 1),
        "prediction": round(score, 4),
        "bridge": "ztf-real-bogus@1",
    })

# In production: df = pd.read_parquet("my_results/", dtype_backend="pyarrow")
df_results = pd.DataFrame(rows)

threshold = 0.7
n_real = int((df_results["prediction"] >= threshold).sum())
print(f"Real (>= {threshold}): {n_real}  |  Bogus: {15 - n_real}")
display(df_results)
